# Clinical ICD Coding using ClinicalBERT
## Baseline vs Negation-Aware vs Section-Aware Models
This notebook loads saved models/predictions, computes evaluation metrics, and generates publication-quality visualizations.

## 1. Imports

In [ ]:

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix
)

print("ALL IMPORTS LOADED")


## 2. Load Saved Labels and Predictions

In [ ]:

with open("val_labels.pkl", "rb") as f:
    val_labels = pickle.load(f)

with open("baseline_predictions.pkl", "rb") as f:
    baseline_preds = pickle.load(f)

with open("negation_predictions.pkl", "rb") as f:
    negation_preds = pickle.load(f)

with open("section_predictions.pkl", "rb") as f:
    section_preds = pickle.load(f)

print("ALL LABELS AND PREDICTIONS LOADED")


## 3. Compute Evaluation Metrics

In [ ]:

models = ["Baseline", "Negation-Aware", "Section-Aware"]

results = {}

prediction_sets = {
    "Baseline": baseline_preds,
    "Negation-Aware": negation_preds,
    "Section-Aware": section_preds
}

for model_name, preds in prediction_sets.items():

    micro_precision = precision_score(
        val_labels,
        preds,
        average="micro",
        zero_division=0
    ) * 100

    micro_recall = recall_score(
        val_labels,
        preds,
        average="micro",
        zero_division=0
    ) * 100

    micro_f1 = f1_score(
        val_labels,
        preds,
        average="micro",
        zero_division=0
    ) * 100

    macro_precision = precision_score(
        val_labels,
        preds,
        average="macro",
        zero_division=0
    ) * 100

    macro_recall = recall_score(
        val_labels,
        preds,
        average="macro",
        zero_division=0
    ) * 100

    macro_f1 = f1_score(
        val_labels,
        preds,
        average="macro",
        zero_division=0
    ) * 100

    hamming = hamming_loss(
        val_labels,
        preds
    )

    results[model_name] = {
        "Micro Precision": micro_precision,
        "Micro Recall": micro_recall,
        "Micro F1": micro_f1,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Hamming Loss": hamming
    }

comparison_df = pd.DataFrame(results).T

comparison_df


## 4. Save Metrics Table

In [ ]:

comparison_df.to_csv(
    "final_model_comparison.csv"
)

print("FINAL METRICS CSV SAVED")


## 5. Overall Model Performance Comparison

In [ ]:

micro_f1 = comparison_df["Micro F1"].values
macro_f1 = comparison_df["Macro F1"].values

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8,5))

ax.bar(
    x - width/2,
    micro_f1,
    width,
    label="Micro F1"
)

ax.bar(
    x + width/2,
    macro_f1,
    width,
    label="Macro F1"
)

ax.set_ylabel("Score (%)")
ax.set_title("Overall Model Performance Comparison")

ax.set_xticks(x)
ax.set_xticklabels(models)

ax.legend()

plt.tight_layout()

plt.savefig(
    "micro_macro_f1_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 6. Precision vs Recall Comparison

In [ ]:

micro_precision = comparison_df["Micro Precision"].values
micro_recall = comparison_df["Micro Recall"].values

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8,5))

ax.bar(
    x - width/2,
    micro_precision,
    width,
    label="Precision"
)

ax.bar(
    x + width/2,
    micro_recall,
    width,
    label="Recall"
)

ax.set_ylabel("Score (%)")
ax.set_title("Precision vs Recall Comparison")

ax.set_xticks(x)
ax.set_xticklabels(models)

ax.legend()

plt.tight_layout()

plt.savefig(
    "precision_recall_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 7. Performance Heatmap

In [ ]:

plt.figure(figsize=(8,5))

sns.heatmap(
    comparison_df[
        [
            "Micro Precision",
            "Micro Recall",
            "Micro F1",
            "Macro Precision",
            "Macro Recall",
            "Macro F1"
        ]
    ],
    annot=True,
    cmap="Blues",
    fmt=".2f"
)

plt.title("Performance Heatmap")

plt.tight_layout()

plt.savefig(
    "performance_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 8. ICD Label Distribution

In [ ]:

label_counts = val_labels.sum(axis=0)

plt.figure(figsize=(12,5))

plt.bar(
    range(len(label_counts)),
    label_counts
)

plt.xlabel("ICD Label Index")
plt.ylabel("Frequency")

plt.title("ICD Label Distribution")

plt.tight_layout()

plt.savefig(
    "icd_label_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 9. Validation Loss Comparison

In [ ]:

validation_loss = [
    0.1574,
    0.1669,
    0.1736
]

plt.figure(figsize=(7,5))

plt.bar(
    models,
    validation_loss
)

plt.ylabel("Validation Loss")

plt.title("Validation Loss Comparison")

plt.tight_layout()

plt.savefig(
    "validation_loss_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 10. Confusion Matrix for Most Frequent ICD Label

In [ ]:

mcm = multilabel_confusion_matrix(
    val_labels,
    baseline_preds
)

label_idx = np.argmax(
    val_labels.sum(axis=0)
)

cm = mcm[label_idx]

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(
    "Confusion Matrix for Most Frequent ICD Label"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

plt.savefig(
    "confusion_matrix_top_icd.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 11. Prediction Error Distribution

In [ ]:

baseline_errors = np.sum(
    np.abs(val_labels - baseline_preds)
)

negation_errors = np.sum(
    np.abs(val_labels - negation_preds)
)

section_errors = np.sum(
    np.abs(val_labels - section_preds)
)

plt.figure(figsize=(7,5))

plt.bar(
    models,
    [
        baseline_errors,
        negation_errors,
        section_errors
    ]
)

plt.ylabel("Total Prediction Errors")

plt.title("Prediction Error Comparison")

plt.tight_layout()

plt.savefig(
    "prediction_error_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 12. Qualitative Prediction Example

In [ ]:

sample_idx = 10

print("TRUE LABELS:")
print(np.where(val_labels[sample_idx] == 1)[0])

print("\nBASELINE PREDICTIONS:")
print(np.where(baseline_preds[sample_idx] == 1)[0])

print("\nNEGATION-AWARE PREDICTIONS:")
print(np.where(negation_preds[sample_idx] == 1)[0])

print("\nSECTION-AWARE PREDICTIONS:")
print(np.where(section_preds[sample_idx] == 1)[0])


## Notebook Completed